# FAST-UAV - Tutorial
*Author: Félix Pollet - 2023* <br>

전기 무인 항공기의 다분야 설계를 위한 프레임워크인 FAST-UAV에 오신 것을 환영합니다.
FAST-UAV는 항공기 설계를 위한 수치적 기능을 제공하는 오픈 소스 라이브러리 [FAST-OAD](https://fast-oad.readthedocs.io)를 기반으로 합니다. FAST-OAD의 계산 핵심은 그 자체로 [OpenMDAO 프레임워크](https://openmdao.org/)를 기반으로 합니다.

이 노트는 FAST-UAV의 기본 사용법을 소개하는 것을 목표로 합니다. 이 튜토리얼이 끝나면 다음을 수행할 수 있습니다:
1. FAST-UAV에서 UAV 설계 문제 설정하기
2. 최적화를 실행하여 주어진 사양 세트에 대해 가장 실현 가능한 설계를 얻습니다.
3. 결과 시각화하기

FAST-UAV의 작동 방식은 FAST-OAD와 매우 유사하다는 점에 유의하세요. 기본 사항에 대해 자세히 알아보려면 [FAST-OAD 문서](https://fast-oad.readthedocs.io)를 참조하세요.

## 1. Setting up and analyzing the initial problem

### a) 폴더 및 파일

작업을 정리하기 위해 `데이터/`와 `작업 디렉토리/`라는 두 개의 다른 폴더를 사용하겠습니다.

데이터/` 폴더에는 구성 파일, 미션 파일, 소스(참조) 파일이 저장됩니다. workdir/` 폴더에는 임시 파일(예: 문제 입력 및 출력의 사본)이 저장됩니다. 다른 파일에 대해 자세히 설명하겠습니다:

* 'configuration.yaml' 파일은 최적화 또는 평가할 UAV 컨셉(멀티로터, 고정익 또는 하이브리드), 최적화 변수/제약 조건/목표, 최적화 알고리즘 설정 등 설계 문제를 정의할 수 있는 최소한의 인터페이스로서, 이 파일에는 최적화 변수/제약 조건/목표가 포함되어 있습니다. 기본 설정 파일은 주요 UAV 개념과 설계 문제에 대해 FAST-UAV에서 제공하며, `데이터/설정` 폴더에서 찾을 수 있습니다.

* 설계 문제 정의에는 `missions.yaml` 파일이 필요합니다. 이 파일을 사용하면 임무 프로필을 정의할 수 있습니다(예: 상승 단계에 이어 순항 및 호버링이 있는 경우). 여기서 이 정의는 순전히 정성적인 것입니다. 각 비행 구간과 관련된 값(예: 상승 속도 및 호버링 지속 시간)은 아래에 설명된 `problem_inputs.xml` 파일에 제공됩니다.

* 입력 파일 `problem_inputs.xml`을 사용하면 설계 사양(예: 임무 요구 사항 및 프로펠러 수)에 대한 값을 지정하고 고급 사용자의 경우 모델 매개 변수에 액세스하여 미세 조정할 수 있습니다. 실제로는 원시 XML 파일을 열거나 수정하지 않고도 입력 파일에 액세스하고 수정할 수 있는 작은 인터페이스가 제공됩니다(이 노트북의 아래 내용 참조). FAST-UAV는 'data/source_files' 디렉터리에서 찾을 수 있는 몇 가지 입력 파일 예제를 제공합니다.

### b) 사례 연구
이 튜토리얼의 목표는 사용자가 정의한 일련의 사양이 주어졌을 때 헥사콥터 UAV의 최적 크기를 찾는 것입니다. 예를 들어, 최적화하고자 하는 멀티로터 UAV의 성능 사양은 설계 페이로드 5.5kg, 18분 체공 시간, 추력 대 중량비 1.97 등 [DJI 매트릭스 600 프로](https://www.dji.com/matrice600-pro)의 사양과 유사합니다.

다음 셀에서는 필요한 라이브러리를 가져오고 폴더와 파일의 경로를 선언합니다.

In [1]:
# Import required librairies
import os.path as pth
import openmdao.api as om
import logging
import warnings
import shutil
import fastoad.api as oad
from time import time
import matplotlib.pyplot as plt
from fastuav.utils.postprocessing.analysis_and_plots import *

# Declare paths to folders and files
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir"
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_DJI_M600.xml")

# For having log messages display on screen
#logging.basicConfig(level=logging.INFO, format="%(levelname)-8s: %(message)s")
#warnings.filterwarnings(action="ignore")

# For using all screen width
from IPython.display import display, HTML, IFrame
display(HTML("<style>.container { width:95% !important; }</style>"))

c:\Users\user\.conda\envs\FastUAV\lib\site-packages\stdatm\__init__.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


디자인 프로세스에 관련된 모델 간의 구조와 링크를 보여주는 [N2 다이어그램](http://openmdao.org/twodocs/versions/latest/basic_guide/make_n2.html) 시각화도 유용한 기능입니다:

In [ ]:
N2_FILE = pth.join(WORK_FOLDER_PATH, "n2.html")
oad.write_n2(CONFIGURATION_FILE, N2_FILE, overwrite=True)
IFrame(src=N2_FILE, width="100%", height="500px")

이제 디자인 문제의 입력값을 살펴보겠습니다. 먼저 문제에 대한 입력 파일을 DJI M600 사양의 소스(참조) 파일(`problem_inputs_DJI_M600.xml`)의 복사본으로 생성합니다:

In [ ]:
oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)

이제 생성된 [입력 파일](./workdir/problem_inputs.xml)을 체크아웃할 수 있습니다. 이 파일의 값은 사용자가 수정할 수 있으며 계산 프로세스를 실행할 때 FAST-UAV에 의해 고려됩니다.<br>.
변수 뷰어`는 입력 파일의 내용을 검사하고 수정할 수 있는 방법을 제공합니다. 표 위의 드롭다운 목록을 통해 표시된 변수를 필터링할 수 있습니다.

변수는 네 가지 주요 카테고리로 그룹화됩니다:
* *'미션'* 카테고리에는 크기 조정 미션(그리고 노트북 [1b_Mission_performance](1b_Mission_performance.ipynb)에서 볼 수 있듯이 다른 모든 설계 외 미션과 관련된 모든 변수(예: 설계 페이로드 질량 및 호버 내구성 요구 사항)가 포함됩니다.
* *'모델'* 카테고리는 설계 프로세스를 지원하는 모델의 매개변수와 관련이 있습니다. 이러한 매개변수의 미세 조정은 스케일링 법칙과 회귀 모델에 익숙한 고급 사용자에게만 권장됩니다.
* '최적화'* 카테고리에는 최적화 프로세스와 관련된 모든 변수, 즉 최적화 도구와 제약 조건에 따라 변경되는 디자인 변수가 포함됩니다.
* 마지막으로 *'데이터'* 카테고리에는 다양한 작동 조건에서의 프로펠러 직경 및 회전 속도와 같이 구성 요소 특성 및 성능과 관련된 나머지 모든 변수가 포함됩니다.

값을 자유롭게 수정하고(예: *"mission:sizing:payload:mass"* 변수로 지정된 페이로드 질량 요구 사항), 더 진행하기 전에 저장 버튼을 클릭하세요.

In [ ]:
INPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
oad.variable_viewer(INPUT_FILE)

## 2. Running the design optimization
이제 최적화 프로세스를 실행할 수 있습니다. 기본 [구성 파일](./data/configurations/multirotor_mdo.yaml)에는 질량 최소화 목표가 정의되어 있습니다. 즉, 요구 사항(예: 18분의 호버링 내구성)을 충족하는 가장 가벼운 디자인을 추구합니다.

In [ ]:
optim_problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)

모든 것이 잘 되었다면 이전 셀의 출력에 "최적화 완료"가 표시되어야 합니다. FAST-UAV가 생성한 임시 출력 파일을 복사하여 최적화 결과를 `data/` 폴더에 영구적으로 저장해 봅시다:

In [ ]:
OUTPUT_FILE = pth.join(WORK_FOLDER_PATH, "problem_outputs.xml")
DJI_M600_OUTPUT_FILE = pth.join(SOURCE_FOLDER_PATH, 'problem_outputs_DJI_M600_mdo.xml')
shutil.copy(OUTPUT_FILE, DJI_M600_OUTPUT_FILE)

*참고: 고급 사용자는 최적화 목표를 범위/지구력 최대화 목표 등으로 대체할 수 있습니다. 이렇게 하려면 최적화 목표와 관련된 구성 파일의 마지막 줄을 다음과 같이 수정해야 합니다.

```yaml
optimization:
  objective:
    #- name: data:weights:mtow  # mass minimization objective
    
    - name: data:performance:range:cruise  # range maximization objective
      scaler: -1  # this multiplier defines a maximization objective instead of a minimization 
```

*An example of optimization with a range maximization objective is provided in notebook [2_FixedWing_Design](2_FixedWing_Design.ipynb).*

## 3. Results and analysis
최적화 과정의 결과를 분석하는 방법에는 여러 가지가 있습니다.

첫째, `optimizer_viewer`는 수렴 시 최적화 변수, 제약 조건 및 목적 함수를 편리하게 요약하여 제공합니다. 설계 변수나 제약 조건에 활성 바운드가 있는 경우 노란색으로 강조 표시되고, 이를 위반한 경우 빨간색으로 강조 표시됩니다.

In [ ]:
oad.optimization_viewer(CONFIGURATION_FILE)

둘째, '변수 뷰어' 도구를 사용하여 .xml 출력 파일을 로드하여 시스템의 모든 변수에 대한 최적화 결과를 확인할 수 있습니다:

In [ ]:
oad.variable_viewer(OUTPUT_FILE)

마지막으로 최적의 설계의 기하학적 구조와 질량 분석을 표시하는 몇 가지 함수가 제공됩니다:

In [ ]:
fig = multirotor_geometry_plot(DJI_M600_OUTPUT_FILE)
fig.show()

In [ ]:
fig = mass_breakdown_sun_plot_drone(DJI_M600_OUTPUT_FILE)
fig.show()

## 4. Mission Scenario Visualization

시나리오 프로파일(미션 요구사항 및 비행 프로파일)을 시각화하여 설계 요구사항을 더 잘 이해할 수 있습니다.

In [ ]:
# problem_inputs.xml에서 시나리오 데이터 읽기
from fastoad.io.variable_io import DataFile
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# XML 파일에서 데이터 읽기
variables = DataFile(INPUT_FILE)
scenario_data = variables.to_dataframe()

# 미션 관련 변수 추출
mission_vars = scenario_data[scenario_data['name'].str.contains('mission:', na=False)]

print("=== 시나리오 프로파일 변수 ===")
for idx, row in mission_vars.iterrows():
    print(f"{row['name']}: {row['val']} {row['units'] if row['units'] else ''}")

In [ ]:
# 시나리오 프로파일 시각화
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Sizing Mission Profile', 'Payload Requirements',
        'Hover Endurance', 'Cruise Speed & Range',
        'Climb/Descent Profile', 'Energy Requirements'
    ),
    specs=[
        [{"type": "scatter"}, {"type": "bar"}],
        [{"type": "bar"}, {"type": "bar"}],
        [{"type": "scatter"}, {"type": "bar"}]
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.15
)

# 헬퍼 함수: 변수 값 가져오기
def get_var_value(df, var_name, default=0.0):
    """DataFrame에서 변수 값을 가져오는 함수"""
    result = df[df['name'] == var_name]
    if not result.empty:
        return result.iloc[0]['val']
    return default


In [ ]:

# 1) Sizing Mission Profile (시간별 고도 프로파일 - 개념도)
# Cruise altitude 가져오기
cruise_alt = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:altitude', 0.0)
if cruise_alt == 0:
    # Hover 미션인 경우 기본 고도 설정
    cruise_alt = 100.0
    print(f"Note: This is a hover mission. Using nominal altitude: {cruise_alt} m")
else:
    print(f"Cruise altitude: {cruise_alt} m")


In [ ]:

# 실제 시나리오: Climb -> Cruise -> Hover -> Descent
phases = ['Takeoff', 'Climb', 'Cruise', 'Hover', 'Descent', 'Landing']
time_phases = [0, 2, 5, 20, 38, 40, 42]  # 예시 시간 (분)
altitude_phases = [0, 0, cruise_alt, cruise_alt, cruise_alt, cruise_alt, 0]

fig.add_trace(
    go.Scatter(
        x=time_phases, 
        y=altitude_phases, 
        mode='lines+markers',
        name='Altitude',
        line=dict(color='blue', width=2),
        marker=dict(size=8)
    ),
    row=1, col=1
)

# 2) Payload Requirements
payload_mass = get_var_value(scenario_data, 'mission:sizing:payload:mass', 0.0)
payload_power = get_var_value(scenario_data, 'mission:sizing:payload:power:multirotor', 0.0)

fig.add_trace(
    go.Bar(
        x=['Mass (kg)', 'Power (W)'],
        y=[payload_mass, payload_power],
        marker_color=['#FF6B6B', '#4ECDC4'],
        name='Payload',
        text=[f'{payload_mass:.1f}', f'{payload_power:.1f}'],
        textposition='outside'
    ),
    row=1, col=2
)

# 3) Hover Endurance Requirement
hover_duration = get_var_value(scenario_data, 'mission:sizing:main_route:hover:duration', 0.0)

fig.add_trace(
    go.Bar(
        x=['Hover Duration'],
        y=[hover_duration],  # 이미 분 단위
        marker_color='#95E1D3',
        name='Hover',
        text=[f'{hover_duration:.1f} min'],
        textposition='outside'
    ),
    row=2, col=1
)

# 4) Cruise Speed & Range
cruise_speed = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:speed:multirotor', 0.0)
cruise_distance = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:distance', 0.0)

fig.add_trace(
    go.Bar(
        x=['Speed (m/s)', 'Distance (km)'],
        y=[cruise_speed, cruise_distance / 1000],  # km로 변환
        marker_color=['#F38181', '#AA96DA'],
        name='Cruise',
        text=[f'{cruise_speed:.1f}', f'{cruise_distance/1000:.2f}'],
        textposition='outside'
    ),
    row=2, col=2
)

# 5) Climb/Descent Profile
climb_rate = get_var_value(scenario_data, 'mission:sizing:main_route:climb:rate:multirotor', 0.0)
# descent rate는 데이터에 없으므로 climb rate와 동일하다고 가정
descent_rate = climb_rate

phases_vertical = ['Climb Rate', 'Descent Rate']
rates = [climb_rate, descent_rate]

fig.add_trace(
    go.Scatter(
        x=phases_vertical,
        y=rates,
        mode='markers',
        marker=dict(size=15, color=['green', 'orange']),
        name='Vertical Speed',
        text=[f'{r:.1f} m/s' for r in rates],
        textposition='top center'
    ),
    row=3, col=1
)

# 6) Energy Requirements (계산된 값이 있다면)
energy_req = get_var_value(scenario_data, 'mission:sizing:main_route:energy', 0.0)
if energy_req > 0:
    energy_wh = energy_req / 3600  # Wh로 변환
else:
    energy_wh = 0

fig.add_trace(
    go.Bar(
        x=['Energy Required'],
        y=[energy_wh],
        marker_color='#FCBAD3',
        name='Energy',
        text=[f'{energy_wh:.0f} Wh'],
        textposition='outside'
    ),
    row=3, col=2
)

# 레이아웃 업데이트
fig.update_xaxes(title_text="Time (min)", row=1, col=1)
fig.update_yaxes(title_text="Altitude (m)", row=1, col=1)
fig.update_yaxes(title_text="Value", row=1, col=2)
fig.update_yaxes(title_text="Duration (min)", row=2, col=1)
fig.update_yaxes(title_text="Value", row=2, col=2)
fig.update_yaxes(title_text="Rate (m/s)", row=3, col=1)
fig.update_yaxes(title_text="Energy (Wh)", row=3, col=2)

fig.update_layout(
    height=1000,
    showlegend=False,
    title_text="Mission Scenario Profile - Design Requirements",
    title_font_size=16
)

fig.show()


In [ ]:
# 상세 미션 타임라인 플롯
fig2 = go.Figure()

# 실제 시나리오 데이터를 기반으로 타임라인 구성
hover_time = get_var_value(scenario_data, 'mission:sizing:main_route:hover:duration', 0.0) * 60  # 분을 초로 변환
cruise_dist = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:distance', 0.0)
cruise_spd = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:speed:multirotor', 0.0)

# Cruise time 계산 (0으로 나누기 방지)
if cruise_spd > 0 and cruise_dist > 0:
    cruise_time = cruise_dist / cruise_spd
else:
    cruise_time = 0

# Cruise altitude
climb_alt = get_var_value(scenario_data, 'mission:sizing:main_route:cruise:altitude', 0.0)
if climb_alt == 0:
    climb_alt = 100.0  # Hover 미션용 기본값

# Climb/Descent time 계산 (0으로 나누기 방지)
climb_rate = get_var_value(scenario_data, 'mission:sizing:main_route:climb:rate:multirotor', 0.0)
if climb_rate > 0:
    climb_time = climb_alt / climb_rate
else:
    climb_time = climb_alt / 3.0  # 기본 climb rate 3 m/s

# Descent rate는 데이터에 없으므로 climb rate와 동일하다고 가정
descent_rate = climb_rate if climb_rate > 0 else 3.0
descent_time = climb_alt / descent_rate

# 타임라인 구성 (누적 시간, 초 단위)
timeline = {
    'Takeoff': 30,
    'Climb': climb_time,
    'Cruise': cruise_time,
    'Hover': hover_time,
    'Descent': descent_time,
    'Landing': 30
}

cumulative_time = 0
times = [0]
altitudes = [0]
phases_labels = []

for phase, duration in timeline.items():
    if phase == 'Takeoff':
        times.append(cumulative_time + duration)
        altitudes.append(0)
        phases_labels.append(phase)
        cumulative_time += duration
    elif phase == 'Climb':
        times.append(cumulative_time + duration)
        altitudes.append(climb_alt)
        phases_labels.append(phase)
        cumulative_time += duration
    elif phase == 'Cruise':
        if duration > 0:  # Cruise가 있는 경우만
            times.append(cumulative_time + duration)
            altitudes.append(climb_alt)
            phases_labels.append(phase)
            cumulative_time += duration
    elif phase == 'Hover':
        times.append(cumulative_time + duration)
        altitudes.append(climb_alt)
        phases_labels.append(phase)
        cumulative_time += duration
    elif phase == 'Descent':
        times.append(cumulative_time + duration)
        altitudes.append(0)
        phases_labels.append(phase)
        cumulative_time += duration
    elif phase == 'Landing':
        times.append(cumulative_time + duration)
        altitudes.append(0)
        phases_labels.append(phase)

# 시간을 분으로 변환
times_min = [t / 60 for t in times]

fig2.add_trace(go.Scatter(
    x=times_min,
    y=altitudes,
    mode='lines+markers',
    line=dict(color='royalblue', width=3),
    marker=dict(size=10, color='orange'),
    name='Mission Profile',
    fill='tozeroy',
    fillcolor='rgba(65, 105, 225, 0.2)'
))

# 각 페이즈를 텍스트로 표시
for i, (t, alt, label) in enumerate(zip(times_min[1:], altitudes[1:], phases_labels)):
    fig2.add_annotation(
        x=t,
        y=alt + 10,
        text=label,
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor='gray',
        ax=0,
        ay=-30
    )

fig2.update_layout(
    title=f"Detailed Mission Timeline Profile (Total: {times_min[-1]:.1f} min)",
    xaxis_title="Time (minutes)",
    yaxis_title="Altitude (m)",
    hovermode='x unified',
    height=500,
    showlegend=True,
    template='plotly_white'
)

fig2.show()


In [ ]:
# 미션 요구사항 요약 테이블
import pandas as pd

# 주요 미션 변수 추출
mission_summary = {
    'Parameter': [],
    'Value': [],
    'Unit': []
}

# 실제 변수명 사용
key_params = {
    'mission:sizing:payload:mass': ('Payload Mass', 'kg'),
    'mission:sizing:payload:power:multirotor': ('Payload Power', 'W'),
    'mission:sizing:main_route:hover:duration': ('Hover Endurance', 'min'),
    'mission:sizing:main_route:cruise:speed:multirotor': ('Cruise Speed', 'm/s'),
    'mission:sizing:main_route:cruise:distance': ('Cruise Distance', 'm'),
    'mission:sizing:main_route:cruise:altitude': ('Cruise Altitude', 'm'),
    'mission:sizing:main_route:climb:rate:multirotor': ('Climb Rate', 'm/s'),
    'mission:sizing:thrust_weight_ratio:multirotor': ('Thrust-to-Weight Ratio', '-'),
}

for key, (param_name, unit) in key_params.items():
    value = get_var_value(scenario_data, key, None)
    if value is not None:
        # 단위 변환
        if unit == 'm' and value > 1000:
            value = value / 1000
            unit = 'km'
        
        mission_summary['Parameter'].append(param_name)
        mission_summary['Value'].append(f'{value:.2f}')
        mission_summary['Unit'].append(unit)

df_summary = pd.DataFrame(mission_summary)

print("\n=== Mission Requirements Summary ===")
print(df_summary.to_string(index=False))

# Plotly 테이블로도 표시
fig3 = go.Figure(data=[go.Table(
    header=dict(
        values=['<b>Parameter</b>', '<b>Value</b>', '<b>Unit</b>'],
        fill_color='paleturquoise',
        align='left',
        font=dict(size=14, color='black')
    ),
    cells=dict(
        values=[df_summary.Parameter, df_summary.Value, df_summary.Unit],
        fill_color='lavender',
        align='left',
        font=dict(size=12)
    )
)])

fig3.update_layout(
    title="Mission Requirements Summary Table",
    height=400
)

fig3.show()
